# Week 2 — 자세 분류 (룰 기반)
**환경**: Python 3.10.11 / Windows / VSCode Jupyter

**동작**
- Azure Kinect Body Tracking SDK로 관절 좌표 추출
- 어깨-엉덩이 각도 계산 → Supine / Lateral_L / Lateral_R / Prone 판정
- 분류 결과를 CSV에 타임스탬프와 함께 저장
- Kinect 없을 때: 저장된 이미지에 더미 관절 데이터로 분류 테스트 가능

**실행 순서**: 셀을 위에서부터 순서대로 실행하세요 (Shift+Enter)

## 0. 패키지 설치

In [ ]:
%pip install opencv-python numpy pandas matplotlib

## 1. 라이브러리 로드 & Kinect 확인

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import sys
from datetime import datetime
from pathlib import Path
sys.path.insert(0, "..")
from utils.db import insert_posture

# Kinect Body Tracking SDK 연결 시도
try:
    import pykinect_azure as pykinect
    KINECT_AVAILABLE = True
    print("[OK] pykinect_azure 로드 성공")
except ImportError:
    KINECT_AVAILABLE = False
    print("[오프라인 모드] pykinect_azure 없음 → 더미 데이터로 분류 테스트")

print(f"pandas  버전: {pd.__version__}")
print(f"OpenCV  버전: {cv2.__version__}")

## 2. 설정값

In [ ]:
# =============================================
#  설정값
# =============================================

INTERVAL_SEC     = 60     # 정기 촬영 간격 (초)
MOTION_THRESHOLD = 30     # 움직임 감지 임계값

SAVE_BASE_DIR = Path("./sleep_frames")
CSV_DIR       = Path("./results")
CSV_DIR.mkdir(parents=True, exist_ok=True)

today    = datetime.now().strftime("%Y%m%d")
SAVE_DIR = SAVE_BASE_DIR / today
SAVE_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = CSV_DIR / f"{today}_posture_log.csv"

print(f"이미지 저장 폴더: {SAVE_DIR.resolve()}")
print(f"CSV 저장 경로  : {CSV_PATH.resolve()}")

## 3. 자세 분류 원리 설명

Azure Kinect Body Tracking SDK가 반환하는 관절 중 4개를 사용합니다.

```
SHOULDER_LEFT  (왼쪽 어깨)
SHOULDER_RIGHT (오른쪽 어깨)
HIP_LEFT       (왼쪽 엉덩이)
HIP_RIGHT      (오른쪽 엉덩이)
NOSE           (코) ← Supine vs Prone 구분용
```

**분류 로직**
```
1. 어깨 중심 ~ 엉덩이 중심을 잇는 벡터를 계산
2. 이 벡터가 수평(X축)에 가까우면 → Lateral (옆으로 누운)
   수직(Y축)에 가까우면 → Supine 또는 Prone
3. Supine vs Prone은 코(NOSE)의 Z값으로 구분
   코가 카메라 쪽(Z 작음)이면 Supine, 반대면 Prone
4. Lateral은 왼쪽/오른쪽 어깨 높이 차이로 L/R 구분
```

## 4. 자세 분류 함수 정의

In [ ]:
# 자세 상수
POSTURE_SUPINE    = "Supine"     # 천장 보고 눕기
POSTURE_PRONE     = "Prone"      # 엎드려 눕기
POSTURE_LATERAL_L = "Lateral_L"  # 왼쪽으로 눕기
POSTURE_LATERAL_R = "Lateral_R"  # 오른쪽으로 눕기
POSTURE_UNKNOWN   = "Unknown"    # 판단 불가

# 분류 각도 임계값 (도 단위)
LATERAL_ANGLE_THRESHOLD = 45  # 이 값보다 작으면 Lateral 판정


def classify_posture(joints: dict) -> tuple[str, float]:
    """
    관절 좌표 딕셔너리를 받아 자세를 분류합니다.

    입력 형식:
        joints = {
            'SHOULDER_LEFT':  (x, y, z),
            'SHOULDER_RIGHT': (x, y, z),
            'HIP_LEFT':       (x, y, z),
            'HIP_RIGHT':      (x, y, z),
            'NOSE':           (x, y, z),   # 없으면 생략 가능
        }

    반환:
        (자세 레이블, 신뢰도 각도)
    """
    required = ['SHOULDER_LEFT', 'SHOULDER_RIGHT', 'HIP_LEFT', 'HIP_RIGHT']
    if not all(k in joints for k in required):
        return POSTURE_UNKNOWN, 0.0

    sl = np.array(joints['SHOULDER_LEFT'])
    sr = np.array(joints['SHOULDER_RIGHT'])
    hl = np.array(joints['HIP_LEFT'])
    hr = np.array(joints['HIP_RIGHT'])

    # 어깨 중심 ~ 엉덩이 중심 벡터
    shoulder_mid = (sl + sr) / 2
    hip_mid      = (hl + hr) / 2
    body_vec     = shoulder_mid - hip_mid

    # XZ 평면에서 몸통 벡터의 수평 각도 계산
    # X: 좌우, Y: 상하, Z: 앞뒤 (카메라 기준)
    angle_from_vertical = np.degrees(
        np.arctan2(abs(body_vec[0]), abs(body_vec[1]))
    )

    # ── Lateral 판정 ────────────────────────────────
    # 몸통이 수평에 가까울수록(angle_from_vertical 작을수록) Lateral
    if angle_from_vertical < LATERAL_ANGLE_THRESHOLD:
        # 왼쪽/오른쪽 구분: 어깨 Y값 차이
        # 카메라에서 봤을 때 위쪽 어깨가 어느 쪽인지로 판단
        if sl[1] < sr[1]:  # 왼쪽 어깨가 더 위
            return POSTURE_LATERAL_L, angle_from_vertical
        else:
            return POSTURE_LATERAL_R, angle_from_vertical

    # ── Supine vs Prone 판정 ────────────────────────
    # NOSE가 있으면 코 Z값으로 구분, 없으면 어깨 Z값 평균 사용
    if 'NOSE' in joints:
        nose_z    = joints['NOSE'][2]
        shoulder_z = (sl[2] + sr[2]) / 2
        face_forward = nose_z < shoulder_z  # 코가 카메라 쪽이면 앙와위
    else:
        # NOSE 없을 때: 어깨 Z와 엉덩이 Z 차이로 추정
        shoulder_z   = (sl[2] + sr[2]) / 2
        hip_z        = (hl[2] + hr[2]) / 2
        face_forward = shoulder_z < hip_z

    if face_forward:
        return POSTURE_SUPINE, angle_from_vertical
    else:
        return POSTURE_PRONE, angle_from_vertical


# ── 테스트: 예시 관절 데이터로 확인 ─────────────────
test_cases = [
    {
        "name": "Supine (천장 보고 눕기)",
        "joints": {
            "SHOULDER_LEFT":  (-0.2, 0.5, 1.0),
            "SHOULDER_RIGHT": ( 0.2, 0.5, 1.0),
            "HIP_LEFT":       (-0.2, 0.0, 1.0),
            "HIP_RIGHT":      ( 0.2, 0.0, 1.0),
            "NOSE":           ( 0.0, 0.7, 0.8),  # 코가 카메라 방향
        }
    },
    {
        "name": "Prone (엎드려 눕기)",
        "joints": {
            "SHOULDER_LEFT":  (-0.2, 0.5, 1.0),
            "SHOULDER_RIGHT": ( 0.2, 0.5, 1.0),
            "HIP_LEFT":       (-0.2, 0.0, 1.0),
            "HIP_RIGHT":      ( 0.2, 0.0, 1.0),
            "NOSE":           ( 0.0, 0.7, 1.3),  # 코가 카메라 반대
        }
    },
    {
        "name": "Lateral_L (왼쪽으로 눕기)",
        "joints": {
            "SHOULDER_LEFT":  (0.5, 0.8, 1.0),   # 왼쪽 어깨 위
            "SHOULDER_RIGHT": (0.5, 0.2, 1.0),
            "HIP_LEFT":       (0.0, 0.8, 1.0),
            "HIP_RIGHT":      (0.0, 0.2, 1.0),
        }
    },
    {
        "name": "Lateral_R (오른쪽으로 눕기)",
        "joints": {
            "SHOULDER_LEFT":  (0.5, 0.2, 1.0),
            "SHOULDER_RIGHT": (0.5, 0.8, 1.0),   # 오른쪽 어깨 위
            "HIP_LEFT":       (0.0, 0.2, 1.0),
            "HIP_RIGHT":      (0.0, 0.8, 1.0),
        }
    },
]

print("── 분류 함수 테스트 ──")
for tc in test_cases:
    result, angle = classify_posture(tc["joints"])
    match = "✅" if tc["name"].startswith(result) else "❌"
    print(f"{match} [{tc['name']}] → {result}  (각도: {angle:.1f}°)")

## 5. CSV 저장 함수 정의

In [ ]:
def save_to_csv(timestamp: int, posture: str, angle: float,
                label: str, image_path: str):
    """
    분류 결과를 CSV에 한 행씩 추가합니다.
    CSV 컬럼: timestamp / datetime / posture / angle / capture_type / image_path
    """
    row = pd.DataFrame([{
        "timestamp":    timestamp,
        "datetime":     datetime.fromtimestamp(timestamp).strftime("%Y-%m-%d %H:%M:%S"),
        "posture":      posture,
        "angle":        round(angle, 2),
        "capture_type": label,        # 'regular' or 'motion'
        "image_path":   image_path,
    }])

    # 파일 있으면 append, 없으면 새로 생성
    row.to_csv(
        CSV_PATH,
        mode="a",
        header=not CSV_PATH.exists(),
        index=False
    )


def load_csv() -> pd.DataFrame:
    """저장된 CSV 불러오기"""
    if not CSV_PATH.exists():
        return pd.DataFrame()
    return pd.read_csv(CSV_PATH)


print("[OK] CSV 함수 정의 완료")

## 6-A. 실시간 분류 실행 — Kinect 모드
정기 촬영 + 움직임 감지 + 자세 분류를 동시에 실행합니다  
종료: 미리보기 창에서 `q` 키

In [ ]:
if not KINECT_AVAILABLE:
    print("Kinect 미연결 → 6-B 셀(오프라인 분류)을 실행하세요")
else:
    pykinect.initialize_libraries()

    device_config = pykinect.default_configuration
    device_config.color_resolution = pykinect.K4A_COLOR_RESOLUTION_1080P
    device_config.depth_mode       = pykinect.K4A_DEPTH_MODE_NFOV_UNBINNED
    device = pykinect.start_device(config=device_config)

    body_tracker = pykinect.start_body_tracker()

    print("[Kinect] 연결 성공! 자세 분류 시작")
    print("미리보기 창에서 'q' 키를 누르면 종료됩니다\n")

    prev_gray         = None
    last_capture_time = 0
    regular_count     = 0
    motion_count      = 0
    last_posture      = POSTURE_UNKNOWN

    POSTURE_COLOR = {
        POSTURE_SUPINE:    (100, 220, 100),
        POSTURE_PRONE:     (80,  80,  255),
        POSTURE_LATERAL_L: (255, 180,  50),
        POSTURE_LATERAL_R: (255, 120, 200),
        POSTURE_UNKNOWN:   (160, 160, 160),
    }

    try:
        while True:
            capture                = device.update()
            ret_color, color_image = capture.get_color_image()
            ret_depth, depth_image = capture.get_depth_image()

            if not ret_color or not ret_depth:
                time.sleep(0.1)
                continue

            now       = time.time()
            depth_vis = cv2.normalize(
                depth_image, None, 0, 255, cv2.NORM_MINMAX
            ).astype(np.uint8)
            depth_vis = cv2.applyColorMap(depth_vis, cv2.COLORMAP_JET)
            curr_gray = cv2.GaussianBlur(
                cv2.cvtColor(color_image, cv2.COLOR_BGR2GRAY), (21, 21), 0)
            saved     = False

            # ── 관절 추출 & 자세 분류 ────────────────
            body_frame = body_tracker.update()
            joints     = {}
            if body_frame.get_num_bodies() > 0:
                body   = body_frame.get_body(0)
                skel   = body.numpy_joints
                KEY_MAP = {
                    'SHOULDER_LEFT':  26, 'SHOULDER_RIGHT': 12,
                    'HIP_LEFT':       18, 'HIP_RIGHT':       4,
                    'NOSE':            3,
                }
                for name, idx in KEY_MAP.items():
                    joints[name] = tuple(skel[idx, :3])

            posture, angle = classify_posture(joints)
            last_posture   = posture

            # ── 정기 촬영 ────────────────────────────
            if now - last_capture_time >= INTERVAL_SEC:
                ts        = int(now)
                img_path  = str(SAVE_DIR / f"{ts}_regular_color.png")
                cv2.imwrite(img_path, color_image)
                cv2.imwrite(str(SAVE_DIR / f"{ts}_regular_depth.png"), depth_vis)
                insert_posture(ts, posture, angle, "regular", img_path, upload=True)
                regular_count    += 1
                last_capture_time = now
                saved             = True
                print(f"[{datetime.now().strftime('%H:%M:%S')}] "
                      f"[정기 #{regular_count}] {posture} ({angle:.1f}°)")

            # ── 움직임 감지 촬영 ─────────────────────
            if prev_gray is not None:
                score = float(np.mean(cv2.absdiff(prev_gray, curr_gray)))
                if score > MOTION_THRESHOLD and not saved:
                    ts        = int(now)
                    img_path  = str(SAVE_DIR / f"{ts}_motion_color.png")
                    cv2.imwrite(img_path, color_image)
                    cv2.imwrite(str(SAVE_DIR / f"{ts}_motion_depth.png"), depth_vis)
                    insert_posture(ts, posture, angle, "motion", img_path, upload=True)
                    motion_count += 1
                    print(f"[{datetime.now().strftime('%H:%M:%S')}] "
                          f"[움직임 #{motion_count}] {posture} (score={score:.1f})")

            prev_gray = curr_gray

            # ── 미리보기 ─────────────────────────────
            preview = color_image.copy()
            cv2.rectangle(preview, (0, 0), (preview.shape[1], 55), (0,0,0), -1)
            cv2.putText(preview,
                f"자세: {last_posture}  |  정기: {regular_count}  움직임: {motion_count}",
                (12, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                POSTURE_COLOR.get(last_posture, (255,255,255)), 2)

            cv2.imshow("자세 분류 (q: 종료)",
                       cv2.resize(preview, (1280, 720)))

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

            time.sleep(0.5)

    except KeyboardInterrupt:
        print("[종료]")
    finally:
        device.close()
        cv2.destroyAllWindows()
        print("Supabase posture_log 저장 완료")

## 6-B. 오프라인 분류 — 저장된 이미지에 더미 관절 데이터 적용
Kinect 없이 분류 로직과 CSV 저장을 테스트합니다  
실제 Kinect 연동 후에는 6-A만 사용하면 됩니다

In [ ]:
import random

# 더미 관절 데이터 생성 함수
def make_dummy_joints(posture: str) -> dict:
    """테스트용 더미 관절 좌표 생성"""
    base = {
        POSTURE_SUPINE: {
            "SHOULDER_LEFT":  (-0.2, 0.5, 1.0), "SHOULDER_RIGHT": (0.2, 0.5, 1.0),
            "HIP_LEFT":       (-0.2, 0.0, 1.0), "HIP_RIGHT":      (0.2, 0.0, 1.0),
            "NOSE":           ( 0.0, 0.7, 0.8),
        },
        POSTURE_PRONE: {
            "SHOULDER_LEFT":  (-0.2, 0.5, 1.0), "SHOULDER_RIGHT": (0.2, 0.5, 1.0),
            "HIP_LEFT":       (-0.2, 0.0, 1.0), "HIP_RIGHT":      (0.2, 0.0, 1.0),
            "NOSE":           ( 0.0, 0.7, 1.3),
        },
        POSTURE_LATERAL_L: {
            "SHOULDER_LEFT":  (0.5, 0.8, 1.0), "SHOULDER_RIGHT": (0.5, 0.2, 1.0),
            "HIP_LEFT":       (0.0, 0.8, 1.0), "HIP_RIGHT":      (0.0, 0.2, 1.0),
        },
        POSTURE_LATERAL_R: {
            "SHOULDER_LEFT":  (0.5, 0.2, 1.0), "SHOULDER_RIGHT": (0.5, 0.8, 1.0),
            "HIP_LEFT":       (0.0, 0.2, 1.0), "HIP_RIGHT":      (0.0, 0.8, 1.0),
        },
    }
    joints = {}
    for k, v in base[posture].items():
        joints[k] = tuple(x + random.uniform(-0.05, 0.05) for x in v)
    return joints


# 저장된 이미지에 더미 분류 적용
color_files  = sorted(SAVE_DIR.glob("*_color.png"))
posture_pool = [POSTURE_SUPINE, POSTURE_LATERAL_L,
                POSTURE_LATERAL_R, POSTURE_PRONE]

if not color_files:
    print("저장된 이미지 없음")
    print("week2_01 또는 week2_02 노트북을 먼저 실행해서 이미지를 저장하세요")
    print("\n더미 데이터 5개로 Supabase 저장 테스트를 진행합니다...")

    for i in range(5):
        ts      = int(time.time()) + i * 60
        posture = random.choice(posture_pool)
        joints  = make_dummy_joints(posture)
        result, angle = classify_posture(joints)
        insert_posture(ts, result, angle, "regular", f"dummy_{ts}.png", upload=False)
        print(f"  더미 #{i+1}: {result} ({angle:.1f}°)")
else:
    print(f"이미지 {len(color_files)}장에 더미 분류 적용 중...\n")
    for f in color_files:
        parts   = f.stem.split("_")
        ts      = int(parts[0])
        label   = parts[1] if len(parts) > 1 else "regular"
        posture = random.choice(posture_pool)
        joints  = make_dummy_joints(posture)
        result, angle = classify_posture(joints)
        insert_posture(ts, result, angle, label, str(f), upload=True)
        print(f"  [{datetime.fromtimestamp(ts).strftime('%H:%M:%S')}] "
              f"{label:8s} → {result} ({angle:.1f}°)")

print(f"\nSupabase posture_log 저장 완료")

## 7. CSV 결과 확인 & 시각화

In [ ]:
df = load_csv()

if df.empty:
    print("CSV 없음 — 6-A 또는 6-B 셀을 먼저 실행하세요")
else:
    print(f"총 기록 수: {len(df)}개")
    print()
    print(df.tail(10).to_string(index=False))

    print("\n── 자세별 집계 ──")
    counts = df["posture"].value_counts()
    for posture, cnt in counts.items():
        pct = cnt / len(df) * 100
        print(f"  {posture:12s}: {cnt}회 ({pct:.1f}%)")

    # 파이차트
    COLORS = {
        POSTURE_SUPINE:    "#5BBF72",
        POSTURE_LATERAL_L: "#F4A742",
        POSTURE_LATERAL_R: "#E07BB5",
        POSTURE_PRONE:     "#5B8EBF",
        POSTURE_UNKNOWN:   "#AAAAAA",
    }
    colors = [COLORS.get(p, "#CCCCCC") for p in counts.index]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # 파이차트
    axes[0].pie(
        counts.values, labels=counts.index,
        colors=colors, autopct="%1.1f%%",
        startangle=140, pctdistance=0.82
    )
    axes[0].set_title("수면 자세 분포", fontsize=13)

    # 시간대별 자세 변화
    posture_order = [POSTURE_SUPINE, POSTURE_LATERAL_L,
                     POSTURE_LATERAL_R, POSTURE_PRONE, POSTURE_UNKNOWN]
    posture_num   = {p: i for i, p in enumerate(posture_order)}
    df["posture_num"] = df["posture"].map(posture_num)

    scatter_colors = [COLORS.get(p, "#CCCCCC") for p in df["posture"]]
    axes[1].scatter(
        range(len(df)), df["posture_num"],
        c=scatter_colors, s=80, zorder=3
    )
    axes[1].plot(
        range(len(df)), df["posture_num"],
        color="#CCCCCC", linewidth=0.8, zorder=2
    )
    axes[1].set_yticks(range(len(posture_order)))
    axes[1].set_yticklabels(posture_order)
    axes[1].set_xlabel("촬영 순서")
    axes[1].set_title("시간대별 자세 변화", fontsize=13)
    axes[1].grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()